In [1]:
import sys
import os

sys.path.append("/home/datalab/nfs/deepfm/")

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import pandas as pd
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
import json
from tqdm import tqdm

from src.datasets import StreamDataset
from src.datasets.collate import collate_fn
from src.models import DeepFM

In [3]:
kt_features = [    
    "inn_kt_index",
    "okved_cd_kt_index",
    "okato_cd_kt_index",
    "bic_kt_34_index",
    "bic_kt_56_index",
    "bic_kt_79_index",
    "num_kt_13_index",
    "num_kt_45_index",
    "num_kt_68_index",
    "okved_cd_kt_lvl1_index",
    "okved_cd_kt_lvl2_index",
    "okved_cd_kt_lvl3_index"]

kt_double_features = [
    "kt_avg_sum",
    "kt_stddev_sum",
    "kt_min_sum",
    "kt_max_sum",
    "kt_median_sum",
    "kt_skewness_sum",
    "kt_buyers_count"]

dt_features =  [    
    "inn_dt_index",
    "okved_cd_dt_index",
    "okato_cd_dt_index",
    "bic_dt_34_index",
    "bic_dt_56_index",
    "bic_dt_79_index",
    "num_dt_13_index",
    "num_dt_45_index",
    "num_dt_68_index",
    "okved_cd_dt_lvl1_index",
    "okved_cd_dt_lvl2_index",
    "okved_cd_dt_lvl3_index"]

dt_double_features = [
    "dt_avg_sum",
    "dt_stddev_sum",
    "dt_min_sum",
    "dt_max_sum",
    "dt_median_sum",
    "dt_skewness_sum",
    "dt_buyers_count"]

label_column = "label"

In [4]:
import gzip, pickle

with gzip.open("/home/datalab/nfs/deepfm/data/train_21/test_dict.pkl.gz", "rb") as f:
    test_dict = pickle.load(f)

In [5]:
# # test = pd.read_parquet('/home/datalab/nfs/deepfm/data/train_10_coalesced/test_2024-10-19_2024-10-19.parquet')

# test_path = "/home/datalab/nfs/deepfm/data/train_11_coalesced/test_2024-10-20_2024-10-22.parquet"
# test_dataset = StreamDataset(
#     file_path=test_path,
#     dt_features=dt_features,
#     kt_features=kt_features,
#     dt_double_features=dt_double_features,
#     kt_double_features=kt_double_features,
#     label_column=label_column,
#     chunk_size=4096
# )

# dataloader = DataLoader(
#     dataset=test_dataset,
#     batch_size=4096,
#     collate_fn=collate_fn
# )

In [5]:
device = torch.device("cuda")

with open("/home/datalab/nfs/deepfm/data/train_21/user_feature_sizes.json", "r") as f:
    user_feature_sizes = json.load(f)

with open("/home/datalab/nfs/deepfm/data/train_21/item_feature_sizes.json", "r") as f:
    item_feature_sizes = json.load(f)

model = DeepFM(
    embed_dim=128,
    num_user_double_feats=7,
    num_item_double_feats=7,
    user_feature_sizes=user_feature_sizes,
    item_feature_sizes=item_feature_sizes,
).to(device)

checkpoint = torch.load("/home/datalab/nfs/deepfm/deepfm_logs/train_21_boevoy_zapusk_vse_fichi/model_best.pth", device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

DeepFM(
  (user_embed): FeatureEmbedding(
    (embeddings): ModuleDict(
      (inn_dt_index): Embedding(2798745, 128)
      (okved_cd_dt_index): Embedding(2643, 51)
      (okato_cd_dt_index): Embedding(60671, 128)
      (bic_dt_34_index): Embedding(84, 9)
      (bic_dt_56_index): Embedding(56, 7)
      (bic_dt_79_index): Embedding(96, 9)
      (num_dt_13_index): Embedding(14, 3)
      (num_dt_45_index): Embedding(26, 5)
      (num_dt_68_index): Embedding(4, 2)
      (okved_cd_dt_lvl1_index): Embedding(91, 9)
      (okved_cd_dt_lvl2_index): Embedding(101, 10)
      (okved_cd_dt_lvl3_index): Embedding(69, 8)
    )
  )
  (item_embed): FeatureEmbedding(
    (embeddings): ModuleDict(
      (inn_kt_index): Embedding(2364265, 128)
      (okved_cd_kt_index): Embedding(2580, 50)
      (okato_cd_kt_index): Embedding(59481, 128)
      (bic_kt_34_index): Embedding(79, 8)
      (bic_kt_56_index): Embedding(51, 7)
      (bic_kt_79_index): Embedding(96, 9)
      (num_kt_13_index): Embedding(17, 4)
  

Loading auxilary dictionaries (damn thats a lot of memory)

In [6]:
import gzip, pickle

with gzip.open("../data/train_21/dt_embeddings_dict.pkl.gz", "rb") as f: dt_embs = pickle.load(f)
with gzip.open("../data/train_21/kt_embeddings_dict.pkl.gz", "rb") as f: kt_embs = pickle.load(f)
with gzip.open("../data/train_21/dt_features_dict.pkl.gz", "rb") as f: dt_feat = pickle.load(f)
with gzip.open("../data/train_21/kt_features_dict.pkl.gz", "rb") as f: kt_feat = pickle.load(f)

In [10]:
# uid = list(test_dict.keys())[0]

# data = dt_feat.get(uid, {})
# cat_user = torch.tensor([[data.get(f, 0) for f in dt_features]], dtype=torch.long, device="cuda")
# cont_user = torch.tensor([[data.get(f, 0.0) for f in dt_double_features]], dtype=torch.float32, device="cuda")
# emb0_user = torch.tensor(dt_embs.get(uid, np.zeros(256)), dtype=torch.float32, device="cuda").unsqueeze(0)

# # make_dot(model.embed_user(cat, cont, emb0), params=dict(model.named_parameters())).render("dt", format="png")

# iid = list(kt_feat.keys())[0]
# cat_item = torch.tensor([[data.get(f, 0) for f in kt_features]], dtype=torch.long, device="cuda")
# cont_item = torch.tensor([[data.get(f, 0.0) for f in kt_double_features]], dtype=torch.float32, device="cuda")
# emb0_item = torch.tensor(kt_embs.get(iid, np.zeros(256)), dtype=torch.float32, device="cuda").unsqueeze(0)

# torch.onnx.export(
#     model,
#     (cat_item, cat_user, cont_item, cont_user, emb0_item, emb0_user),
#     "../deepfm_logs/train_21_boevoy_zapusk_vse_fichi/graph/model_graph.onnx",
#     input_names=["cat_item", "cat_user", "cont_item", "cont_user", "emb0_item", "emb0_user"],
#     output_names=["logits", "kt_weights", "dt_weights"],
#     opset_version=11
# )

Making embeddings

In [13]:
dt_embeddings = {}
dt_attentions = {}

for uid in tqdm(test_dict):
    data = dt_feat.get(uid, {})
    cat = torch.tensor([[data.get(f, 0) for f in dt_features]], dtype=torch.long, device="cuda")
    cont = torch.tensor([[data.get(f, 0.0) for f in dt_double_features]], dtype=torch.float32, device="cuda")
    emb0 = torch.tensor(dt_embs.get(uid, np.zeros(256)), dtype=torch.float32, device="cuda").unsqueeze(0)
    with torch.no_grad():
        emb, attention = model.embed_user(cat, cont, emb0)
    dt_embeddings[uid] = emb.squeeze(0).cpu().numpy()
    dt_attentions[uid] = attention.squeeze(0).cpu().numpy()

100%|██████████| 118998/118998 [02:51<00:00, 692.34it/s]


In [19]:
kt_embeddings = {}
kt_attentions = {}

for iid, data in tqdm(kt_feat.items()):
    try:
        cat = torch.tensor([[data.get(f, 0) for f in kt_features]], dtype=torch.long, device="cuda")
        cont = torch.tensor([[data.get(f, 0.0) for f in kt_double_features]], dtype=torch.float32, device="cuda")
        emb0 = torch.tensor(kt_embs.get(iid, np.zeros(256)), dtype=torch.float32, device="cuda").unsqueeze(0)
        with torch.no_grad():
            emb, attention = model.embed_item(cat, cont, emb0)
        kt_embeddings[iid] = emb.squeeze(0).cpu().numpy()
        kt_attentions[iid] = attention.squeeze(0).cpu().numpy()
    except Exception as e:
        print(f"\n\nError processing item {iid}: {e}\nData: {data}\n\n")
        continue

100%|██████████| 2364265/2364265 [58:31<00:00, 673.34it/s]  


In [7]:
# dt_embeddings = {}
# dt_attentions = {}
# kt_embeddings = {}
# kt_attentions = {}

# with torch.no_grad():
#     for batch in tqdm(dataloader):
#         user = batch["user"].to(device)
#         item = batch["item"].to(device)
#         double_user = batch["double_user"].to(device)
#         double_item = batch["double_item"].to(device)
#         label = batch["label"].to(device)

#         u_dt, attn_dt = model.embed_user(user, double_user)
#         u_kt, attn_kt = model.embed_item(item, double_item)

#         user_ids = user[:, 0].cpu().numpy()
#         item_ids = item[:, 0].cpu().numpy()
        
#         for i in range(len(user_ids)):
#             uid, iid = user_ids[i], item_ids[i]
#             if uid not in dt_embeddings:
#                 dt_embeddings[uid] = u_dt[i].cpu().numpy()
#                 dt_attentions[uid] = attn_dt[i].cpu().numpy()
#             if iid not in kt_embeddings:
#                 kt_embeddings[iid] = u_kt[i].cpu().numpy()
#                 kt_attentions[iid] = attn_kt[i].cpu().numpy()


3261it [16:48,  3.23it/s]


In [20]:
import gzip
import pickle

def save_compressed(path, embeddings):
    with gzip.open(path, "wb") as f:
        pickle.dump(embeddings, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("saved")

dt_embeds_path = "../data/train_21/embeddings/dt_embeddings_test.pkl.gz"
kt_embeds_path = "../data/train_21/embeddings/kt_embeddings_test.pkl.gz"

dt_attns_path = "../data/train_21/attentions/dt_attentions_test.pkl.gz"
kt_attns_path = "../data/train_21/attentions/kt_attentions_test.pkl.gz"

# сохраняем наши словарики
# save_compressed(dt_embeds_path, dt_embeddings)
save_compressed(kt_embeds_path, kt_embeddings)
# save_compressed(dt_attns_path, dt_attentions)
save_compressed(kt_attns_path, kt_attentions)

saved
saved


In [22]:
len(kt_embeddings)

2364265